In [3]:
import json
with open("../../data/sample_preprocessed.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(data[0])

{'id': 0, 'text': '我刚刚开始record', 'tokens': ['我', '刚', '刚', '开', '始', 'record']}


In [6]:
def detect_language(token):
    for char in token:
        if 0x4e00 <= ord(char) <= 0x9fff:
            return "ZH"
    return "EN"

#test
print(detect_language("record"))   # EN
print(detect_language("我"))        # ZH
print(detect_language("hello"))    # EN
print(detect_language("始"))        # ZH


EN
ZH
EN
ZH


In [7]:
def labels_list(item):
    labels = []
    for string in item["tokens"]:
        labels.append(detect_language(string))
    item["labels"] = labels
    return item

print(labels_list(data[0]))

{'id': 0, 'text': '我刚刚开始record', 'tokens': ['我', '刚', '刚', '开', '始', 'record'], 'labels': ['ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'EN']}


In [8]:
for item in data:
    labels_list(item)
    print(item["tokens"])
    print(item["labels"])
    print()

['我', '刚', '刚', '开', '始', 'record']
['ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'EN']

['嗯', 'hello', '我', '的', '名', '字', '叫', '徐', '妍']
['ZH', 'EN', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH']

['嗯', '初', '次', '见', '面', 'nice', 'to', 'meet', 'you', '嗯']
['ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'EN', 'EN', 'EN', 'EN', 'ZH']

['今', '天', '呢', '我', '非', '常', '希', '望', '能', '够', '通', '过', '这', '个', '机', '会', '去', '跟', '你', 'make', 'friends']
['ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'EN', 'EN']

['嗯', '你', '知', '道', '就', '是']
['ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH']



In [ ]:
def find_switch_points(labels):
    indices = []
    for i in range(len(labels)):
        if i > 0:
            if labels[i] != labels[i-1]:
                indices.append(i)
    return indices

def add_switch_points(item):
    item["switch_points"] = find_switch_points(item["labels"])
    return item

for item in data:
    add_switch_points(item)
    print(item["tokens"])
    print(item["labels"])
    print(item["switch_points"])
    print()

with open("../../data/sample_switch_points.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

['我', '刚', '刚', '开', '始', 'record']
['ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'EN']
[5]

['嗯', 'hello', '我', '的', '名', '字', '叫', '徐', '妍']
['ZH', 'EN', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH']
[1, 2]

['嗯', '初', '次', '见', '面', 'nice', 'to', 'meet', 'you', '嗯']
['ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'EN', 'EN', 'EN', 'EN', 'ZH']
[5, 9]

['今', '天', '呢', '我', '非', '常', '希', '望', '能', '够', '通', '过', '这', '个', '机', '会', '去', '跟', '你', 'make', 'friends']
['ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'EN', 'EN']
[19]

['嗯', '你', '知', '道', '就', '是']
['ZH', 'ZH', 'ZH', 'ZH', 'ZH', 'ZH']
[]



In [12]:
from datasets import load_dataset
dataset = load_dataset("CAiRE/ASCEND")


c:\Users\sihan\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sihan\.cache\huggingface\hub\datasets--CAiRE--ASCEND. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 1130/1130 [00:00<00:00, 3158.04 examples/s]


In [18]:
import sys
sys.path.append("../../")
from preprocessing.tokenizer import simple_tokenize

dataset = load_dataset("CAiRE/ASCEND")
dataset = dataset.remove_columns(["audio"])

train_processed = []
validation_processed = []
test_processed = []

for item in dataset["train"]:
    text = item["transcription"]
    tokens = simple_tokenize(text)
    labels = [detect_language(t) for t in tokens]
    switch_points = find_switch_points(labels)
    train_processed.append({
        "id": item["id"],
        "text": text,
        "tokens": tokens,
        "labels": labels,
        "switch_points": switch_points
    })
for item in dataset["validation"]:
    text = item["transcription"]
    tokens = simple_tokenize(text)
    labels = [detect_language(t) for t in tokens]
    switch_points = find_switch_points(labels)
    validation_processed.append({
        "id": item["id"],
        "text": text,
        "tokens": tokens,
        "labels": labels,
        "switch_points": switch_points
    })
for item in dataset["test"]:
    text = item["transcription"]
    tokens = simple_tokenize(text)
    labels = [detect_language(t) for t in tokens]
    switch_points = find_switch_points(labels)
    test_processed.append({
        "id": item["id"],
        "text": text,
        "tokens": tokens,
        "labels": labels,
        "switch_points": switch_points
    })

with open("../../data/train_processed.json", "w", encoding="utf-8") as f:
    json.dump(train_processed, f, ensure_ascii=False, indent=2)
with open("../../data/validation_processed.json", "w", encoding="utf-8") as f:
    json.dump(validation_processed, f, ensure_ascii=False, indent=2)
with open("../../data/test_processed.json", "w", encoding="utf-8") as f:
    json.dump(test_processed, f, ensure_ascii=False, indent=2)